# Путь 1

In [1]:
!pip install pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 79.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pymorphy3]


In [2]:
import re
import math
import pandas as pd
import pymorphy3
from functools import lru_cache
from typing import List, Tuple, Dict, FrozenSet
from collections import defaultdict

загрузка графа

In [3]:
GRAPH_CSV = "edge_final_probabilities.csv"
LAMBDA = -11.72

morph = pymorphy3.MorphAnalyzer()

@lru_cache(maxsize=100_000)
def lemmatize_word(w: str) -> str:
    w = w.lower().strip()
    if not w or not re.search(r"[а-яё]", w):
        return w
    return morph.parse(w)[0].normal_form

def lemmatize_text(text: str) -> List[str]:
    tokens = re.findall(r"[а-яёa-z0-9]+", text.lower())
    return [lemmatize_word(t) for t in tokens if t]

def load_graph(csv_path: str) -> Dict[FrozenSet[str], float]:
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    graph = {}
    for _, row in df.iterrows():
        a = str(row["agent_1"]).strip().lower()
        b = str(row["agent_2"]).strip().lower()
        try:
            p = float(row["probability"])
        except (ValueError, TypeError):
            continue
        if p <= 0:
            continue
        graph[frozenset({a, b})] = p
    print(f"Загружено рёбер в граф: {len(graph)}")
    return graph

GRAPH = load_graph(GRAPH_CSV)

Загружено рёбер в граф: 285572


score анекдота

In [4]:
def extract_candidate_edges(lemmas: List[str]) -> List[FrozenSet[str]]:
    cands = []
    for i in range(len(lemmas) - 1):
        cands.append(frozenset({lemmas[i], lemmas[i + 1]}))
    for i in range(len(lemmas) - 1):
        cands.append(frozenset({f"{lemmas[i]} {lemmas[i+1]}", lemmas[i]}))
        cands.append(frozenset({f"{lemmas[i]} {lemmas[i+1]}", lemmas[i+1]}))
    return cands

In [5]:
def compute_score(joke: str, graph: Dict[FrozenSet[str], float]) -> Tuple[float, int]:
    """
    Возвращает (score, n_edges_used).
    score = mean(ln p) по найденным рёбрам.
    Если рёбер нет — возвращаем (-inf, 0).
    """
    lemmas = lemmatize_text(joke)
    if len(lemmas) < 2:
        return float("-inf"), 0

    log_probs = []
    for cand in extract_candidate_edges(lemmas):
        if cand in graph:
            log_probs.append(math.log(graph[cand]))

    if not log_probs:
        return float("-inf"), 0
    return sum(log_probs) / len(log_probs), len(log_probs)

In [6]:
def classify(score: float) -> str:
    if score == float("-inf"):
        return "no_edges"
    return "joke" if score < LAMBDA else "literary"

генерация анекдота (нейросетевая)

In [7]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

GEN_MODEL = "sberbank-ai/rugpt3small_based_on_gpt2"

def load_generator():
    tok = GPT2Tokenizer.from_pretrained(GEN_MODEL)
    mod = GPT2LMHeadModel.from_pretrained(GEN_MODEL)
    mod.eval()
    if torch.cuda.is_available():
        mod = mod.to("cuda")
    return tok, mod

FEW_SHOT = (
    "Вовочка приходит домой и говорит: — Мама, я получил двойку. "
    "— Почему? — Потому что учитель спросил, сколько будет дважды два, а я сказал четыре. "
    "— Ну и что? — А он спросил, почему не пять.\n"
    "Мужик приходит к врачу: — Доктор, у меня провалы в памяти. "
    "— И давно это у вас? — Что именно?\n"
    "Анекдот:"
)

def generate_jokes(prompts: List[str], tok, mod, num_return: int = 3) -> List[str]:
    jokes = []
    for prompt in prompts:
        full_prompt = FEW_SHOT + " " + prompt
        input_ids = tok.encode(full_prompt, return_tensors="pt")
        if torch.cuda.is_available():
            input_ids = input_ids.to("cuda")
        with torch.no_grad():
            out = mod.generate(
                input_ids,
                max_new_tokens=60,
                num_return_sequences=num_return,
                do_sample=True,
                temperature=0.95,
                top_p=0.92,
                repetition_penalty=1.25,
                pad_token_id=tok.eos_token_id,
            )
        for seq in out:
            text = tok.decode(seq, skip_special_tokens=True)
            text = text.split("Анекдот:")[-1].strip()
            text = text.split("\n")[0].strip()
            text = re.sub(r"\s+", " ", text)
            if len(text.split()) >= 5:
                jokes.append(text)
    return list(dict.fromkeys(jokes))

фильтрация и вывод

In [8]:
def run_pipeline(prompts: List[str], threshold: float = LAMBDA) -> pd.DataFrame:
    tok, mod = load_generator()
    raw = generate_jokes(prompts, tok, mod, num_return=3)
    rows = []
    for j in raw:
        score, n = compute_score(j, GRAPH)
        label = classify(score) if threshold == LAMBDA else (
            "joke" if score < threshold else "literary"
        )
        rows.append({"joke": j, "score": score, "n_edges": n, "label": label})
    return pd.DataFrame(rows)

def display_jokes(df: pd.DataFrame):
    jokes = df[df["label"] == "joke"].sort_values("score")
    print("=" * 70)
    print(f"СМЕШНЫЕ (score < λ={LAMBDA}): {len(jokes)} шт. — оцените вручную")
    print("=" * 70)
    for i, row in enumerate(jokes.itertuples(), 1):
        print(f"\n[{i}] score={row.score:.3f} | рёбер={row.n_edges}")
        print(f"    {row.joke}")
    print("\n" + "=" * 70)
    print(f"ВСЕГО сгенерировано: {len(df)} | "
          f"joke={len(jokes)} | literary={(df['label']=='literary').sum()} | "
          f"no_edges={(df['label']=='no_edges').sum()}")

запуск

In [9]:
if __name__ == "__main__":
    prompts = [
        "Купил мужик шляпу, а она ему как раз.",
        "Приходит бабка к врачу:",
        "Вовочка на уроке:",
    ]
    df = run_pipeline(prompts, threshold=LAMBDA)
    display_jokes(df)
    df.to_csv("generated_jokes_scored.csv", index=False)

tokenizer_config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.71M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.27M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/574 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  551MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sberbank-ai/rugpt3small_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


СМЕШНЫЕ (score < λ=-11.72): 0 шт. — оцените вручную

ВСЕГО сгенерировано: 9 | joke=0 | literary=8 | no_edges=1


In [10]:
res = pd.read_csv('generated_jokes_scored.csv')
res

,joke,score,n_edges,label
0,"Купил мужик шляпу, а она ему как раз. - Ты ког...",-6.214608,1,literary
1,"Купил мужик шляпу, а она ему как раз. Но стоит...",-6.214608,1,literary
2,"Купил мужик шляпу, а она ему как раз. Так вот ...",-6.214608,4,literary
3,Приходит бабка к врачу: — Добрый день доктор! ...,-3.843867,3,literary
4,"Приходит бабка к врачу:— Извините, доктор! Я в...",-3.824056,3,literary
5,Приходит бабка к врачу: - Добрый день доктор! ...,-3.402107,2,literary
6,"Вовочка на уроке: «Сон», учительница: - Здравс...",-4.924692,3,literary
7,Вовочка на уроке: - Вы слышали анекдот про Вов...,-inf,0,no_edges
8,"Вовочка на уроке: «— Как вы думаете, зачем мне...",-6.210600,1,literary


# Путь 2

генерация по графу → восстановление форм → скоринг

In [11]:
!pip install pymorphy3 pymorphy2-dicts-ru pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 92.6 MB/s  0:00:00


In [13]:
import importlib
import sys
import pandas as pd

# модули из приложенных .py
import генерация_предложений_по_графу as gen_graph
import surface_realization as surf

# если модуль менялся — перезагрузить
importlib.reload(gen_graph)
importlib.reload(surf)

print("gen_graph OK, узлов:", len(gen_graph.load_graph(gen_graph.FILE_PATH)))

Автор видит умирать почерпать жизнь быть попадаться дерево застилать ее..
Автор означает к матери быть пресмыкаться написать на родиться броситься ходить с пристань..
Агитатор есть это объявить своё написать на случаться обречь с заморенный..
Агитатор выйти слово не быть соответствовать рот не прийти правом работать лед..
Агитатор сделать мир знать мочь бывать на свет позволить узнать повторять мелькать ничтожество..
Автор видит умирать почерпать жизнь быть попадаться дерево застилать ее..
Автор означает к матери быть пресмыкаться написать на родиться броситься ходить с пристань..
Агитатор есть это объявить своё написать на случаться обречь с заморенный..
Агитатор выйти слово не быть соответствовать рот не прийти правом работать лед..
Агитатор сделать мир знать мочь бывать на свет позволить узнать повторять мелькать ничтожество..
gen_graph OK, узлов: 1287


In [14]:
import random

def generate_lemma_chain(graph, min_words=6, max_words=10, max_steps=30):
    """
    Собирает цепочку ЛЕММ по графу (без meaning — только вершины).
    Возвращает список лемм или None.
    """
    if not graph:
        return None
    current = random.choice(list(graph.keys()))
    tokens = current.split()
    for _ in range(max_steps):
        if len(tokens) >= max_words:
            break
        edges = graph.get(current, [])
        if not edges:
            break
        edge = random.choice(edges)
        tokens.extend(edge["target"].split())
        current = edge["target"]
    if min_words <= len(tokens) <= max_words:
        return tokens
    return None


def generate_lemma_chains(graph, n=200, min_words=6, max_words=10,
                          max_attempts=20000, seed=None):
    if seed is not None:
        random.seed(seed)
    results = []
    seen = set()
    for _ in range(max_attempts):
        if len(results) >= n:
            break
        chain = generate_lemma_chain(graph, min_words=min_words, max_words=max_words)
        if chain is None:
            continue
        key = " ".join(chain)
        if key in seen:
            continue
        seen.add(key)
        results.append(chain)
    return results

In [15]:
GRAPH2 = gen_graph.load_graph(gen_graph.FILE_PATH)
print("Узлов в графе:", len(GRAPH2))

lemma_chains = generate_lemma_chains(
    GRAPH2,
    n=100,
    min_words=6,
    max_words=10,
    seed=42,
)
print("Цепочек-лемм:", len(lemma_chains))

realized = []
for i, lemmas in enumerate(lemma_chains, 1):
    try:
        sent = surf.realize_sentence(lemmas, preserve_order=True)
    except Exception as e:
        sent = f"<<ОШИБКА: {e}>>"
    realized.append({
        "lemmas": " ".join(lemmas),
        "sentence": sent,
    })

df_realized = pd.DataFrame(realized)
df_realized.head(10)

Узлов в графе: 1287
Цепочек-лемм: 100


,lemmas,sentence
0,шлюха лошадь время палец весь тело утро мать с...,Шлюха лошадь время пальцем всё тело утром мать...
1,лекция испуг отношение мать речь божество,Лекция испуг отношение матерью речь божеством.
2,крытый этот свой время свой это дерево необход...,Это своё время своё это дерево необходимостью ...
3,кобель необходимость это время глаз прегрешени...,Кобель необходимость это временем глаза прегре...
4,хозяин пример этот свой слово горький,Хозяин пример этот свой слово горьким.
5,избавитель весь этот место неприятность сознан...,Избавитель весь этот место неприятность сознан...
6,природа время свой человек мать право право не...,Природа время своя человека матерью право прав...
7,сербский другой свой этот отношение стадо самы...,Своё это отношение стадо самое склад этот благ...
8,богач этот свой материал это смерть жизнь трак...,Богач этот свой материал это смертью жизнь тра...
9,сербский свой день утро брамин горький,Сербский свой день утро брамин горьким.


In [16]:
rows = []
for row in df_realized.itertuples(index=False):
    score, n_edges = compute_score(row.sentence, GRAPH)
    label = classify(score)
    rows.append({
        "lemmas": row.lemmas,
        "sentence": row.sentence,
        "score": score,
        "n_edges": n_edges,
        "label": label,
    })

df_scored2 = pd.DataFrame(rows)
df_scored2.to_csv("path2_graph_realized_scored.csv", index=False, encoding="utf-8-sig")
df_scored2["label"].value_counts()

,count
label,
literary,100


In [17]:
df_view = df_scored2.rename(columns={"sentence": "joke"})
display_jokes(df_view)

СМЕШНЫЕ (score < λ=-11.72): 0 шт. — оцените вручную

ВСЕГО сгенерировано: 100 | joke=0 | literary=100 | no_edges=0
